# Figure: Training Trajectories under Multiplicative Gaussian Input Noise

Trains a two-layer ReLU MLP on synthetic regression with multiplicative input noise $\mathbf{x} \leftarrow \mathbf{x} \odot \mathbf{c}$, $\mathbf{c}\sim\mathcal{N}(\mathbf{1},\kappa^2\mathbf{I})$, for several values of $\kappa$, and plots the clean training loss versus iteration. Demonstrates linear convergence to a $\kappa$-controlled error ball (Theorem 5.5).

**Requirements:** PyTorch, GPU optional. Pre-cached CIFAR-10 splits (`targetTrain.npz`, `targetTest.npz`) expected under `data/CIFAR10/Preprocessed/`.


In [ ]:
# =========================
# 3 plots: test accuracy vs kappa
#   (1) MLPNet (attack pipeline)
#   (2) CNNNet (attack pipeline)
#   (3) ImprovedCNNNet (test-only curve)
# MG noise: gauss only (train-time): x * (1 + kappa * N(0,1))
# =========================

# (removed for repo) from google.colab import drive
# (removed for repo) drive.mount(...)
import os, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

# -------------------------
# Output folder
# -------------------------
OUT_DIR = "../figures"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

# -------------------------
# Reproducibility
# -------------------------
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -------------------------
# Load cached pipeline split
# -------------------------
DATA_FOLDER = "../data"
pre_path = Path(DATA_FOLDER) / "CIFAR10" / "Preprocessed"
target_train_npz = pre_path / "targetTrain.npz"
target_test_npz  = pre_path / "targetTest.npz"

def load_npz(path):
    with np.load(path) as f:
        return [f[f"arr_{i}"] for i in range(len(f.files))]

assert target_train_npz.exists() and target_test_npz.exists(), \
    f"Missing {target_train_npz} or {target_test_npz}. Run pipeline preprocessing first."

X_train, y_train = load_npz(str(target_train_npz))
X_test,  y_test  = load_npz(str(target_test_npz))

y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

# -------------------------
# Models
# -------------------------
class CNNNet(nn.Module):
    """
    Attack pipeline CNN:
    conv5x5 (pad same) -> maxpool2 -> conv5x5 (valid) -> maxpool2 -> tanh FC -> logits
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        _, C, H, W = n_in

        self.conv1 = nn.Conv2d(C, 32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 32, kernel_size=5, padding=0)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2)

        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = self.pool1(self.relu1(self.conv1(dummy)))
            x = self.pool2(self.relu2(self.conv2(x)))
            flat_dim = x.numel()

        self.fc = nn.Linear(flat_dim, n_hidden)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.tanh(self.fc(x))
        return self.out(x)

class MLPNet(nn.Module):
    """
    Attack pipeline MLP:
    flatten -> Dense(tanh, n_hidden) -> logits
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        D = n_in[1]
        self.fc = nn.Linear(D, n_hidden)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        x = self.tanh(self.fc(x))
        return self.out(x)

class ImprovedCNNNet(nn.Module):
    """
    Your improved CNN (deeper, BN, dropout).
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        _, C, H, W = n_in

        self.conv1 = nn.Conv2d(C, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(0.2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(0.3)

        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = F.relu(self.bn1(self.conv1(dummy)))
            x = F.relu(self.bn2(self.conv2(x)))
            x = self.pool1(x)
            x = F.relu(self.bn3(self.conv3(x)))
            x = F.relu(self.bn4(self.conv4(x)))
            x = self.pool2(x)
            flat_dim = x.numel()

        self.fc = nn.Linear(flat_dim, n_hidden)
        self.drop3 = nn.Dropout(0.5)
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc(x))
        x = self.drop3(x)
        return self.out(x)

# -------------------------
# Batching + evaluation
# -------------------------
def iterate_minibatches(inputs, targets, batch_size, shuffle=True):
    idx = np.arange(len(inputs))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(inputs), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield inputs[batch_idx], targets[batch_idx]

@torch.no_grad()
def eval_test_acc(model, X, y, batch_size):
    model.eval()
    preds = []
    for xb_np, yb_np in iterate_minibatches(X, y, batch_size, shuffle=False):
        xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        preds.append(pred)
    preds = np.concatenate(preds, axis=0)
    return float(accuracy_score(y, preds))

# -------------------------
# Train + return clean test accuracy
# -------------------------
def train_and_get_test_acc(model_name, kappa, *, epochs, batch_size, lr, l2, n_hidden, seed):
    set_seed(seed)
    n_out = int(len(np.unique(y_train)))

    if model_name == "mlp":
        Xtr = X_train.reshape(X_train.shape[0], -1).astype(np.float32)
        Xte = X_test.reshape(X_test.shape[0], -1).astype(np.float32)
        net = MLPNet(n_in=Xtr.shape, n_hidden=n_hidden, n_out=n_out).to(device)
        train_X, train_y_local = Xtr, y_train
        test_X,  test_y_local  = Xte, y_test

    elif model_name == "cnn":
        net = CNNNet(n_in=X_train.shape, n_hidden=n_hidden, n_out=n_out).to(device)
        train_X, train_y_local = X_train, y_train
        test_X,  test_y_local  = X_test, y_test

    elif model_name == "improved_cnn":
        net = ImprovedCNNNet(n_in=X_train.shape, n_hidden=n_hidden, n_out=n_out).to(device)
        train_X, train_y_local = X_train, y_train
        test_X,  test_y_local  = X_test, y_test

    else:
        raise ValueError("model_name must be one of: mlp, cnn, improved_cnn")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=l2)

    for ep in range(epochs):
        net.train()
        for xb_np, yb_np in iterate_minibatches(train_X, train_y_local, batch_size, shuffle=True):
            xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
            yb = torch.from_numpy(yb_np).to(device=device, dtype=torch.long)

            # MG gauss only
            if kappa > 0.0:
                xb = xb * (1.0 + kappa * torch.randn_like(xb))

            optimizer.zero_grad()
            loss = criterion(net(xb), yb)
            loss.backward()
            optimizer.step()

    return eval_test_acc(net, test_X, test_y_local, batch_size)

# -------------------------
# Plot helper
# -------------------------
def plot_test_curve(kappas, test_accs, out_file):
    fig, ax = plt.subplots(figsize=(6.8, 4.4))
    ax.plot(kappas, test_accs, marker="o", linewidth=2)
    # ax.set_xlabel("MG noise strength κ")
    # ax.set_ylabel("Test Accuracy (clean)")
    plt.xlabel("kappa (κ)")
    plt.ylabel("Final test accuracy (%)")
    #ax.set_title(title)
    ax.grid(True, alpha=0.3)

    out_path = os.path.join(OUT_DIR, out_file)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print("Saved:", out_path)
    plt.show()

# -------------------------
# Settings
# -------------------------
KAPPAS = [0.0, 0.2, 0.5, 0.7, 1.0, 1.25]
EPOCHS = 80

# "Attack pipeline-ish" defaults (you can adjust)
BATCH_SIZE = 128
LR = 1e-3
N_HIDDEN = 100
L2 = 1e-7

# For improved CNN, often better with slightly different hyperparams;
IMPROVED_BATCH = 128
IMPROVED_L2    = 1e-5
IMPROVED_HID   = 100
IMPROVED_LR    = 1e-3
IMPROVED_EPOCHS= 80


# -------------------------
# Improved CNN plot (TEST ONLY)
# -------------------------
improved_test = []
print("\n=== Improved CNN test accuracy vs κ (TEST ONLY) ===")
for k in KAPPAS:
    te = train_and_get_test_acc(
        "improved_cnn", k,
        epochs=IMPROVED_EPOCHS,
        batch_size=IMPROVED_BATCH,
        lr=IMPROVED_LR,
        l2=IMPROVED_L2,
        n_hidden=IMPROVED_HID,
        seed=SEED
    )
    improved_test.append(te)
    print(f"Improved CNN | κ={k:<4} | test(clean)={te*100:.2f}%")

improved_test_pct = [acc * 100 for acc in improved_test]

plot_test_curve(
    #title=f"CNN test accuracy vs κ (epochs={IMPROVED_EPOCHS})",
    kappas=KAPPAS,
    test_accs=improved_test_pct,
    out_file=f"test_acc_vs_kappa_improved_cnn_epochs{IMPROVED_EPOCHS}.png"
)
